<a href="https://colab.research.google.com/github/HesekielJosafat/Tugas-Akhir/blob/main/TA_D3TI_2024_Kel_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisis dan Perbandingan Potensi Halusinasi pada Respons Large Language Model Open-Source terhadap Informasi Wisata Kabupaten Toba Menggunakan Cosine Similarity dan Verifikasi Faktual Berbasis Website

In [3]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
!pip install -qU \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    scikit-learn \
    mistral-common

In [5]:
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -qU ollama sentence-transformers pandas scikit-learn

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pci.ids.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.10.0-2build1_amd64.deb ...
Unpacking libpci3:amd64 (1:3.10.0-2build1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../pciutils_1%3a3.10.0-2build1_amd64.deb ...
Unpacking pciutils (1:3.10.0-2build1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Setting up libpci3:amd64 (1:3.10.0-2bui

In [8]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(15):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("Server Ollama sudah aktif dan siap digunakan!")
            break
    except Exception:
        time.sleep(1)

Server Ollama sudah aktif dan siap digunakan!


In [1]:
import os
import getpass
from google.colab import userdata

try:
    # Try to fetch from Colab Secrets automatically
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as e:
    print(f"Could not automatically fetch HF_TOKEN from Colab Secrets: {e}")
    print("Please enter your Hugging Face Token manually below:")
    HF_TOKEN = getpass.getpass("HF Token: ")

# Set sebagai environment variable agar otomatis dikenali transformers & HF Hub
if HF_TOKEN:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
    os.environ["HF_TOKEN"] = HF_TOKEN
    print(f"HF Token prefix: {HF_TOKEN[:8]}...")
else:
    print("Warning: No HF Token provided. Some private or gated models may not load properly.")

HF Token prefix: hf_cGzrc...


In [9]:
!pip install -q openpyxl
import pandas as pd
from google.colab import files

data = [
    # --- SUMBER WEB & JURNAL DARING ---
    ["Q01", "Pantai Lumban Bulbul", "Wisata", "Tinggi", 7, "Lokasi",
     "Di desa dan kecamatan mana Pantai Lumban Bulbul berada?",
     "Pantai Lumban Bulbul terletak di Jalan Sibalahotang, Desa Lumban Bulbul, Kecamatan Balige, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://jadesta.kemenparekraf.go.id/ (lengkapi link lengkap halaman Desa Wisata Lumban Bulbul)", "", "2026-11-19"],

    ["Q02", "Pantai Lumban Bulbul", "Wisata", "Tinggi", 7, "Deskripsi",
     "Apa ciri khas dan daya tarik utama Pantai Lumban Bulbul?",
     "Pantai Lumban Bulbul terkenal sebagai pantai air tawar berpasir putih di tepi Danau Toba dengan wahana air seperti banana boat dan kano.",
     "Web", "https://jadesta.kemenparekraf.go.id/ (lengkapi link lengkap halaman Desa Wisata Lumban Bulbul)", "", "2026-11-19"],

    ["Q03", "Pantai Lumban Bulbul", "Wisata", "Tinggi", 7, "Fasilitas",
     "Fasilitas apa saja yang tersedia bagi pengunjung di Pantai Lumban Bulbul?",
     "Fasilitas yang tersedia meliputi area parkir, balai pertemuan, kafetaria, kamar mandi umum, kios suvenir, kuliner, musala, dan spot foto.",
     "Web", "https://jadesta.kemenparekraf.go.id/ (lengkapi link lengkap halaman Desa Wisata Lumban Bulbul)", "", "2026-11-19"],

    ["Q04", "Desa Wisata Meat", "Wisata", "Tinggi", 6, "Lokasi",
     "Di kecamatan mana Desa Wisata Meat berada?",
     "Desa Wisata Meat terletak di Kecamatan Tampahan, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://jadesta.kemenparekraf.go.id/desa/meat", "", "2026-11-19"],

    ["Q05", "Desa Wisata Meat", "Wisata", "Tinggi", 6, "Deskripsi",
     "Apa saja daya tarik wisata dan budaya unggulan di Desa Wisata Meat?",
     "Daya tarik utamanya meliputi hamparan persawahan terasering menghadap Danau Toba, kerajinan tenun Ulos tradisional, serta agenda tahunan Event Seribu Tenda.",
     "Web", "https://jadesta.kemenparekraf.go.id/desa/meat | https://doi.org/10.59581/jmpp-widyakarya.v1i4.1469", "", "2026-11-19"],

    ["Q06", "The Kaldera Toba Nomadic Escape", "Wisata", "Tinggi", 8, "Lokasi",
     "Di desa dan kecamatan mana The Kaldera Toba Nomadic Escape berlokasi?",
     "The Kaldera Toba Nomadic Escape terletak di Desa Pardamean Sibisa, Kecamatan Ajibata, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://bpodt.kemenpar.go.id/ (lengkapi link lengkap artikel The Kaldera)", "", "2026-11-19"],

    ["Q07", "The Kaldera Toba Nomadic Escape", "Wisata", "Tinggi", 8, "Deskripsi",
     "Konsep wisata apa yang diusung The Kaldera Toba Nomadic Escape dan desa apa yang terlihat dari tebingnya?",
     "Destinasi kelolaan BPODT ini mengusung konsep wisata nomadik dan glamping di ketinggian sekitar 1.300 mdpl yang menghadap ke ngarai Desa Sigapiton.",
     "Web", "https://bpodt.kemenpar.go.id/ (lengkapi link lengkap artikel The Kaldera)", "", "2026-11-19"],

    ["Q08", "Taman Eden 100", "Wisata", "Tinggi", 6, "Lokasi",
     "Di mana lokasi tepat Taman Eden 100 di Kabupaten Toba?",
     "Taman Eden 100 berada di Desa Lumban Rang, Sionggang Utara, Kecamatan Lumban Julu, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://doi.org/10.55123/toba.v4i4.6523", "", "2026-11-19"],

    ["Q09", "Taman Eden 100", "Wisata", "Tinggi", 6, "Deskripsi",
     "Sejak kapan Taman Eden 100 didirikan dan apa saja daya tarik alam di dalamnya?",
     "Didirikan tahun 2000 dengan luas lebih dari 40 hektare sebagai kawasan ekowisata yang memiliki kebun stroberi, air terjun, Gua Kelelawar, area berkemah, andaliman, dan kantong semar.",
     "Web", "https://doi.org/10.55123/toba.v4i4.6523 | https://doi.org/10.51510/konsep.v3i1.932", "", "2026-11-19"],

    ["Q10", "Air Terjun Situmurun", "Wisata", "Tinggi", 7, "Lokasi",
     "Di desa wisata dan kecamatan mana titik akses Air Terjun Situmurun (Binangalom) berada?",
     "Air Terjun Situmurun terletak di kawasan Desa Wisata Jonggi Nihuta, Kecamatan Lumban Julu, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://jadesta.kemenparekraf.go.id/ (lengkapi link lengkap Desa Wisata Jonggi Nihuta)", "", "2026-11-19"],

    ["Q11", "Air Terjun Situmurun", "Wisata", "Tinggi", 7, "Fasilitas",
     "Moda transportasi apa yang harus digunakan wisatawan untuk mencapai lokasi Air Terjun Situmurun?",
     "Untuk mencapai Air Terjun Situmurun, wisatawan harus menggunakan transportasi air berupa kapal.",
     "Web", "https://jadesta.kemenparekraf.go.id/ (lengkapi link lengkap Desa Wisata Jonggi Nihuta)", "", "2026-11-19"],

    ["Q12", "Bukit Tarabunga", "Wisata", "Tinggi", 5, "Lokasi",
     "Di kecamatan mana Bukit Tarabunga berada di Kabupaten Toba?",
     "Bukit Tarabunga terletak di Desa Tarabunga, Kecamatan Tampahan, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://doi.org/10.34833/panorama.v1i1.1395 | https://doi.org/10.59581/jmpp-widyakarya.v1i4.1476", "", "2026-11-19"],

    ["Q13", "Makam Raja Sisingamangaraja XII", "Wisata", "Tinggi", 6, "Lokasi",
     "Di desa dan kecamatan mana Kompleks Makam Raja Sisingamangaraja XII berada?",
     "Kompleks Makam Raja Sisingamangaraja XII terletak di Desa Pagar Batu, Kecamatan Balige, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://wisata.tobakab.go.id/ (lengkapi link lengkap artikel Makam Sisingamangaraja XII)", "", "2026-11-19"],

    ["Q14", "Bukit Senyum", "Wisata", "Rendah", 3, "Lokasi",
     "Di desa dan kecamatan mana objek wisata Bukit Senyum berada di Kabupaten Toba?",
     "Bukit Senyum terletak di Desa Motung, Kecamatan Ajibata, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://doi.org/10.51827/jiaa.v13i1.244", "", "2026-11-19"],

    ["Q15", "Pantai Pasifik Porsea", "Wisata", "Rendah", 2, "Lokasi",
     "Di desa dan kecamatan mana Pantai Pasifik Porsea berada?",
     "Pantai Pasifik Porsea terletak di Desa Patane IV, Kecamatan Porsea, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://id.wikipedia.org/wiki/Porsea,_Toba (lengkapi jika ada link berita lokal lain)", "", "2026-11-19"],

    ["Q16", "Desa Wisata Sitoluama", "Wisata", "Rendah", 2, "Deskripsi",
     "Apa nama festival panen raya tahunan dan pasar mingguan lokal yang ada di Desa Sitoluama, Kecamatan Laguboti?",
     "Di Desa Sitoluama terdapat agenda budaya tahunan Sitoluama Festival di Jalan Sisingamangaraja dan pasar lokal Sitoluama Sunday Market di Jalan Tugu Pangaribuan.",
     "Web", "https://jadesta.kemenparekraf.go.id/ (lengkapi link lengkap Desa Wisata Sitoluama)", "", "2026-11-19"],

    ["Q17", "Mawar Homestay", "Hotel", "Rendah", 1, "Lokasi",
     "Di desa dan kecamatan mana Mawar Homestay berada di Kabupaten Toba?",
     "Mawar Homestay terletak di Desa Lumban Bulbul, Kecamatan Balige, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://jadesta.kemenparekraf.go.id/ (lengkapi link lengkap halaman Desa Wisata Lumban Bulbul)", "", "2026-11-19"],

    ["Q18", "Labersa Kaldera Resort", "Hotel", "Tinggi", 5, "Lokasi",
     "Di kecamatan mana kawasan pengembangan Labersa Kaldera Resort berada?",
     "Labersa Kaldera Resort dikembangkan di zona otorita The Kaldera, Kecamatan Ajibata, Kabupaten Toba, Sumatera Utara.",
     "Web", "https://bpodt.kemenpar.go.id/ (lengkapi link lengkap siaran pers Labersa Kaldera Resort)", "", "2026-11-19"],

    ["Q19", "Sapadia Hotel Balige", "Hotel", "Rendah", 3, "Lokasi",
     "Di kecamatan mana Sapadia Hotel Balige berada di Kabupaten Toba?",
     "Sapadia Hotel terletak di Kecamatan Balige, Kabupaten Toba, Sumatera Utara.",
     "Web", "Lengkapi link lengkap Traveloka/Tiket.com/situs resmi Sapadia Hotel Balige", "", "2026-11-19"],

    ["Q20", "Hotel Adela Laguboti", "Hotel", "Rendah", 1, "Lokasi",
     "Di kecamatan mana Hotel Adela berada di Kabupaten Toba?",
     "Hotel Adela terletak di Kecamatan Laguboti, Kabupaten Toba, Sumatera Utara.",
     "Web", "Lengkapi link lengkap Traveloka/Booking/Agoda untuk Hotel Adela Laguboti", "", "2026-11-19"],

    # --- SUMBER PDF (TEKS & GAMBAR/TABEL PDF) ---
    ["Q21", "Kawasan Danau Toba (KWU Balige)", "Wisata", "Tinggi", 6, "Deskripsi",
     "Dalam perencanaan Kawasan Wisata Unggulan (KWU) Danau Toba, tema pengembangan apa yang diusung oleh KWU Balige?",
     "Dalam pengembangan kawasan prioritas Danau Toba, KWU Balige mengusung tema pusaka perkotaan yang menampilkan keunikan dan sejarah kota.",
     "PDF (Teks)", "Strategi_Pengembangan_Pariwisata_Kawasan_Danau_Toba.pdf | http://doi.org/10.5614/wpar.2024.22.1.03 - Hal. 4", "", "2026-11-19"],

    ["Q22", "Strategi Pengembangan Danau Toba (Matriks SWOT)", "Wisata", "Tinggi", 6, "Deskripsi",
     "Berdasarkan Tabel Matriks SWOT Strategi Pengembangan Pariwisata Kawasan Danau Toba, apa poin pertama pada strategi Strengths-Opportunities (SO)?",
     "Strategi SO pertama adalah membangun prasarana berupa jalan tol, jalur kereta api, pelabuhan, dan bandara udara yang terintegrasi untuk mendukung konektivitas transportasi antarmoda.",
     "PDF (Gambar)", "Strategi_Pengembangan_Pariwisata_Kawasan_Danau_Toba.pdf | http://doi.org/10.5614/wpar.2024.22.1.03 - Hal. 2 (Tabel 1 Matriks SWOT)", "gambar_dari_pdf/hal2_tabel1_swot.png", "2026-11-19"],

    ["Q25", "Strategi Pengembangan Danau Toba (Matriks SWOT)", "Wisata", "Tinggi", 6, "Deskripsi",
     "Berdasarkan Tabel Matriks SWOT Strategi Pengembangan Pariwisata Kawasan Danau Toba, apa poin pertama pada strategi Strengths-Threats (ST)?",
     "Strategi ST pertama adalah mengembangkan sekolah tinggi pariwisata dalam rangka penyerapan tenaga kerja pariwisata asli daerah.",
     "PDF (Gambar)", "Strategi_Pengembangan_Pariwisata_Kawasan_Danau_Toba.pdf | http://doi.org/10.5614/wpar.2024.22.1.03 - Hal. 2 (Tabel 1 Matriks SWOT)", "gambar_dari_pdf/hal2_tabel1_swot.png", "2026-11-19"],

    # --- SUMBER DATA STATISTIK BPS (CSV / XLSX) ---
    ["Q23", "Statistik Pariwisata Kabupaten Toba", "Wisata", "Tinggi", 5, "Deskripsi",
     "Pada bulan apa kunjungan wisatawan ke Kabupaten Toba mencapai jumlah tertinggi sepanjang tahun 2024 dan berapa totalnya?",
     "Kunjungan wisatawan tertinggi di Kabupaten Toba tahun 2024 terjadi pada bulan Januari dengan total 198.338 wisatawan (27 wisatawan asing dan 198.311 wisatawan nusantara).",
     "CSV/XLSX (BPS)", "Jumlah Wisatawan yang Datang menurut Bulan dan Jenis Wisatawan , 2024.csv (https://tobakab.bps.go.id/)", "", "2026-11-19"],

    ["Q24", "Statistik Akomodasi Kabupaten Toba", "Hotel", "Tinggi", 5, "Deskripsi",
     "Berapa persentase Tingkat Penghunian Kamar (TPK) hotel dan akomodasi lainnya di Kabupaten Toba pada tahun 2022?",
     "Tingkat Penghunian Kamar (TPK) hotel dan akomodasi lainnya di Kabupaten Toba pada tahun 2022 adalah sebesar 29,32 persen.",
     "CSV/XLSX (BPS)", "Tingkat Penghunian Kamar Hotel dan Akomodasi Lainnya, 2022.xlsx (https://tobakab.bps.go.id/)", "", "2026-11-19"],
]

kolom = [
    "id_pertanyaan",
    "destinasi",
    "jenis_objek",
    "kelompok",
    "jumlah_sumber",
    "kategori_pertanyaan",
    "pertanyaan",
    "ground_truth",
    "tipe_sumber",
    "sumber_acuan",
    "gambar_acuan",
    "tanggal_akses",
]

nama_file = "ground truth and question dataset.xlsx"
df = pd.DataFrame(data, columns=kolom)
df = df.sort_values("id_pertanyaan").reset_index(drop=True)
df.to_excel(nama_file, index=False)

print(f"Berhasil membuat '{nama_file}' dengan {len(df)} baris dan {len(kolom)} kolom!")
display(df[["id_pertanyaan", "destinasi", "tipe_sumber", "sumber_acuan", "gambar_acuan"]].head(10))

# Langsung unduh otomatis ke komputer Anda
files.download(nama_file)

Berhasil membuat 'ground truth and question dataset.xlsx' dengan 25 baris dan 12 kolom!


,id_pertanyaan,destinasi,tipe_sumber,sumber_acuan,gambar_acuan
0,Q01,Pantai Lumban Bulbul,Web,https://jadesta.kemenparekraf.go.id/ (lengkapi...,
1,Q02,Pantai Lumban Bulbul,Web,https://jadesta.kemenparekraf.go.id/ (lengkapi...,
2,Q03,Pantai Lumban Bulbul,Web,https://jadesta.kemenparekraf.go.id/ (lengkapi...,
3,Q04,Desa Wisata Meat,Web,https://jadesta.kemenparekraf.go.id/desa/meat,
4,Q05,Desa Wisata Meat,Web,https://jadesta.kemenparekraf.go.id/desa/meat ...,
5,Q06,The Kaldera Toba Nomadic Escape,Web,https://bpodt.kemenpar.go.id/ (lengkapi link l...,
6,Q07,The Kaldera Toba Nomadic Escape,Web,https://bpodt.kemenpar.go.id/ (lengkapi link l...,
7,Q08,Taman Eden 100,Web,https://doi.org/10.55123/toba.v4i4.6523,
8,Q09,Taman Eden 100,Web,https://doi.org/10.55123/toba.v4i4.6523 | http...,
9,Q10,Air Terjun Situmurun,Web,https://jadesta.kemenparekraf.go.id/ (lengkapi...,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import ollama
import pandas as pd
import subprocess, time
from tqdm.auto import tqdm

df_q = pd.read_csv("dataset_pertanyaan.csv")

MODELS = [
    "gemma3:4b",
    "llama3.1:8b",
    "qwen3:8b",
    "ministral-3:8b",
]

SYSTEM_PROMPT = (
    "Anda adalah asisten informasi pariwisata. Jawablah pertanyaan faktual mengenai "
    "destinasi wisata dan penginapan di Kabupaten Toba secara ringkas, jelas, dan akurat "
    "dalam Bahasa Indonesia. Jika Anda tidak mengetahui jawabannya secara pasti, "
    "katakan bahwa Anda tidak tahu."
)

OPTIONS = {
    "temperature": 0.2,
    "top_p": 0.9,
    "num_predict": 300,
    "seed": 42,
}

def pull_dengan_progress_bar(client, model_name):
    """Mengunduh model Ollama sambil menampilkan progress bar hijau Colab."""
    current_digest, bars = "", {}
    try:
        for progress in client.pull(model_name, stream=True):
            digest = progress.get("digest", "")
            if digest != current_digest and current_digest in bars:
                bars[current_digest].close()
            if not digest:
                status = progress.get("status", "")
                if status:
                    tqdm.write(f"  [Ollama] {status}")
                continue
            if digest not in bars and (total := progress.get("total")):
                bars[digest] = tqdm(
                    total=total,
                    desc=f"Download {model_name} ({digest[7:15]})",
                    unit="B",
                    unit_scale=True,
                )
            if completed := progress.get("completed"):
                bars[digest].update(completed - bars[digest].n)
            current_digest = digest
    finally:
        for bar in bars.values():
            bar.close()

hasil = []
client = ollama.Client()

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Memproses Model: {model_name}")
    print(f"{'='*60}")

    try:
        # 1. Download model dengan progress bar hijau
        pull_dengan_progress_bar(client, model_name)

        # 2. Jalankan pertanyaan dengan progress bar hijau
        for _, row in tqdm(df_q.iterrows(), total=len(df_q), desc=f"Uji {model_name}"):
            t_awal = time.time()
            kwargs = {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": row["pertanyaan"]},
                ],
                "options": OPTIONS,
            }
            if "qwen3" in model_name:
                kwargs["think"] = False

            try:
                resp = client.chat(**kwargs)
                jawaban = resp["message"]["content"]
                status_api = "sukses"
            except Exception as e:
                jawaban = ""
                status_api = f"error: {str(e)[:150]}"

            durasi = round(time.time() - t_awal, 2)
            hasil.append({
                "id_pertanyaan": row["id_pertanyaan"],
                "destinasi": row["destinasi"],
                "kelompok": row["kelompok"],
                "pertanyaan": row["pertanyaan"],
                "ground_truth": row["ground_truth"],
                "model": model_name,
                "respons_asli": jawaban,
                "durasi_detik": durasi,
                "status_inferensi": status_api,
            })
            tqdm.write(f"  [{row['id_pertanyaan']}] selesai ({durasi} dtk): {jawaban[:80]}...")

    except Exception as e:
        print(f"ERROR pada model {model_name}: {e}")
        for _, row in df_q.iterrows():
            hasil.append({
                "id_pertanyaan": row["id_pertanyaan"],
                "destinasi": row["destinasi"],
                "kelompok": row["kelompok"],
                "pertanyaan": row["pertanyaan"],
                "ground_truth": row["ground_truth"],
                "model": model_name,
                "respons_asli": "",
                "durasi_detik": 0,
                "status_inferensi": f"error: {str(e)[:150]}",
            })

    finally:
        # Simpan progres ke CSV setiap kali 1 model selesai (aman jika Colab disconnect)
        pd.DataFrame(hasil).to_csv("hasil_respons.csv", index=False)

        # Hapus model dari disk Colab agar penyimpanan tidak penuh
        subprocess.run(["ollama", "rm", model_name], check=False)
        print(f"  Cache disk {model_name} dibersihkan & progres tersimpan.")

print("\nSelesai seluruh model! Semua respons disimpan ke hasil_respons.csv")
pd.read_csv("hasil_respons.csv")[["id_pertanyaan", "model", "status_inferensi", "respons_asli", "durasi_detik"]]


=== Memproses Model: gemma3:4b ===
  [Q1] selesai dalam 131.72 dtk
  [Q2] selesai dalam 0.38 dtk

=== Memproses Model: llama3.1:8b ===
  [Q1] selesai dalam 79.73 dtk
  [Q2] selesai dalam 0.89 dtk

=== Memproses Model: qwen3:8b ===
  [Q1] selesai dalam 40.18 dtk
  [Q2] selesai dalam 0.97 dtk

=== Memproses Model: ministral-3:8b ===
  [Q1] selesai dalam 34.21 dtk
  [Q2] selesai dalam 1.19 dtk

Selesai! Semua respons disimpan ke hasil_respons.csv


,id_pertanyaan,model,respons_asli,durasi_detik
0,Q1,gemma3:4b,Pantai Lumban Bulbul terletak di Desa Harangga...,131.72
1,Q2,gemma3:4b,Museum Batak TB Silalahi Center berada di Keca...,0.38
2,Q1,llama3.1:8b,Pantai Lumban Bulbul terletak di Desa Lumban B...,79.73
3,Q2,llama3.1:8b,Museum Batak TB Silalahi Center berada di Keca...,0.89
4,Q1,qwen3:8b,Pantai Lumban Bulbul berada di Desa Lumban Bul...,40.18
5,Q2,qwen3:8b,Museum Batak TB Silalahi Center berada di Keca...,0.97
6,Q1,ministral-3:8b,Pantai Lumban Bulbul terletak di **Kabupaten T...,34.21
7,Q2,ministral-3:8b,Museum Batak TB Silalahi Center berada di **Ke...,1.19


In [ ]:
import re
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("hasil_respons.csv")

def bersihkan_teks(teks):
    if not isinstance(teks, str):
        return ""
    teks = re.sub(r"<think>.*?</think>", "", teks, flags=re.DOTALL)
    teks = re.sub(r"[*#`>-]+", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks

df["respons_bersih"] = df["respons_asli"].apply(bersihkan_teks)

# Load model embedding sesuai proposal
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

vec_gt = embed_model.encode(df["ground_truth"].fillna("").tolist(), normalize_embeddings=True)
vec_resp = embed_model.encode(df["respons_bersih"].fillna("").tolist(), normalize_embeddings=True)

df["cosine_similarity"] = [
    round(float(cosine_similarity([vec_resp[i]], [vec_gt[i]])[0][0]), 4)
    for i in range(len(df))
]

# Kolom untuk diisi manual saat tahap verifikasi faktual
df["status_verifikasi"] = ""
df["kategori_halusinasi"] = ""

df.to_csv("hasil_evaluasi.csv", index=False)

print("Rata-rata Skor Cosine Similarity per Model:")
print(df.groupby("model")["cosine_similarity"].mean().sort_values(ascending=False))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Rata-rata Skor Cosine Similarity per Model:
model
llama3.1:8b       0.92735
qwen3:8b          0.90795
gemma3:4b         0.87480
ministral-3:8b    0.86055
Name: cosine_similarity, dtype: float64


In [ ]:
import pandas as pd

df = pd.read_csv("hasil_evaluasi.csv")

# 1. Tampilkan perbandingan per pertanyaan secara lengkap di layar
for q_id, grup in df.groupby("id_pertanyaan"):
    print("=" * 80)
    print(f"[{q_id}] {grup.iloc[0]['destinasi']} (Dokumentasi: {grup.iloc[0]['kelompok']})")
    print(f"PERTANYAAN   : {grup.iloc[0]['pertanyaan']}")
    print(f"GROUND TRUTH : {grup.iloc[0]['ground_truth']}")
    print("-" * 80)
    for _, row in grup.iterrows():
        print(f"🤖 {row['model']:15s} | Cosine: {row['cosine_similarity']:.4f} | Waktu: {row['durasi_detik']} dtk")
        print(f"   Jawaban: {row['respons_bersih']}\n")

# 2. Tabel matriks perbandingan skor Cosine Similarity (Model sebagai kolom)
print("\n=== TABEL PERBANDINGAN SKOR COSINE SIMILARITY ===")
tabel_perbandingan = df.pivot(index="id_pertanyaan", columns="model", values="cosine_similarity")
display(tabel_perbandingan)

[Q1] Pantai Lumban Bulbul (Dokumentasi: Tinggi)
PERTANYAAN   : Di mana lokasi Pantai Lumban Bulbul berada dan apa daya tariknya?
GROUND TRUTH : Pantai Lumban Bulbul terletak di Desa Lumban Bulbul, Kecamatan Balige, Kabupaten Toba, Sumatera Utara, terkenal dengan pantai berpasir putih di tepi Danau Toba.
--------------------------------------------------------------------------------
🤖 gemma3:4b       | Cosine: 0.8174 | Waktu: 131.72 dtk
   Jawaban: Pantai Lumban Bulbul terletak di Desa Harangga Ranah, Kecamatan Tarabuang, Kabupaten Toba, Sumatera Utara. Daya tarik utama Pantai Lumban Bulbul: Pasir Putih yang Bersih: Pantai ini terkenal dengan pasirnya yang sangat bersih dan putih, menjadikannya tempat yang ideal untuk bersantai dan berenang. Pemandangan Laut yang Indah: Anda dapat menikmati pemandangan laut Toba yang mempesona. Suasana yang Tenang: Dibandingkan dengan pantai pantai populer lainnya di Toba, Lumban Bulbul cenderung lebih tenang dan sepi. Aktivitas Pantai: Tersedia aktivi

model,gemma3:4b,llama3.1:8b,ministral-3:8b,qwen3:8b
id_pertanyaan,,,,
Q1,0.8174,0.8856,0.8471,0.8522
Q2,0.9322,0.9691,0.8740,0.9637
